# M3 · Reentrenar llantas en una GPU en la nube (Vast.ai)
**IAM-811 · Laboratorio 3**

Sube a `/workspace` este notebook y `m3_entrada.zip`. Ejecuta las celdas en orden.

- Público: [axle](https://universe.roboflow.com/eternal-wiecw/axle-bjpov) y [truck axel](https://universe.roboflow.com/rajan-28noy/truck-axel) (CC BY 4.0). Los 5 puntos salen de la caja de cada llanta.
- Propio: fotos de autos con pseudoetiquetas del M3 base. Las fotos de `test` no se usan para entrenar.
- `ejes_reales.csv`: ejes visibles contados a ojo; **revísalo antes de citarlo en el informe**.

## 1. Entorno y GPU

In [ ]:
import os, sys
!apt-get update -qq && apt-get install -y -qq unzip libgl1 libglib2.0-0 > /dev/null
!{sys.executable} -m pip install -q ultralytics==8.3.240 roboflow==1.1.66 "pillow-heif<1.0"
!{sys.executable} -c "import torch, ultralytics; print('gpu:', torch.cuda.is_available() and torch.cuda.get_device_name(0), '| torch', torch.__version__, '| ultralytics', ultralytics.__version__)"

## 2. Archivos

In [ ]:
!mkdir -p /workspace/m3 && unzip -qo /workspace/m3_entrada.zip -d /workspace/m3 && ls -la /workspace/m3 /workspace/m3/m3_entrada

## 3. Clave de Roboflow y datos
La clave se escribe en el cuadro oculto: no queda guardada en el notebook.

In [ ]:
import os, sys, time, shutil, getpass, subprocess
if not os.environ.get('ROBOFLOW_API_KEY'):
    os.environ['ROBOFLOW_API_KEY'] = getpass.getpass('ROBOFLOW_API_KEY: ').strip()
for intento in range(1, 11):
    for d in ['axle', 'truckaxel']:          # borra descargas a medias
        if not os.path.exists(f'/workspace/m3/descargas/{d}/data.yaml'):
            shutil.rmtree(f'/workspace/m3/descargas/{d}', ignore_errors=True)
    r = subprocess.run([sys.executable, 'm3_reentrenar.py', 'preparar'], cwd='/workspace/m3', capture_output=True, text=True)
    salida = (r.stdout + r.stderr).replace(os.environ['ROBOFLOW_API_KEY'], '***').strip().splitlines()
    if r.returncode == 0:
        for linea in salida:
            if 'it/s' not in linea and '%|' not in linea:
                print(linea)
        print('LISTO')
        break
    print('Intento', intento, 'falló:', salida[-1][:200] if salida else 'sin salida')
    time.sleep(10)


## 4. Entrenar en segundo plano
Puedes cerrar la pestaña, pero **no detengas la instancia**.

In [ ]:
import sys
import subprocess
if subprocess.run(['pgrep', '-f', 'm3_reentrenar.py entrenar'], stdout=subprocess.DEVNULL).returncode == 0:
    print('Ya se está entrenando: no lo lances otra vez.')
else:
    subprocess.Popen([sys.executable, 'm3_reentrenar.py', 'entrenar'], cwd='/workspace/m3', start_new_session=True,
                     stdout=open('/workspace/m3/entrenamiento.log', 'w'), stderr=subprocess.STDOUT)
    print('Entrenamiento lanzado. Mira el avance en la celda siguiente.')

## 5. Ver el avance
Repite esta celda hasta ver `ENTRENAMIENTO TERMINADO`.

In [ ]:
!tail -c 1500 /workspace/m3/entrenamiento.log | tr '\r' '\n' | tail -n 12

## 6. Comparar los modelos

In [ ]:
import sys
!cd /workspace/m3 && {sys.executable} m3_reentrenar.py comparar 2>&1 | tee comparacion.txt

## 7. Archivos para descargar
En el panel de archivos de Jupyter: clic derecho → *Download* sobre `/workspace/m3_resultado.zip`.

In [ ]:
import shutil
os.makedirs('/workspace/m3/resultado', exist_ok=True)
shutil.copy('/workspace/m3/runs/m3_v2/weights/best.pt', '/workspace/m3/resultado/m3_pose_nuevo.pt')
for f in ['results.png', 'results.csv', 'args.yaml']:
    if os.path.exists(f'/workspace/m3/runs/m3_v2/{f}'):
        shutil.copy(f'/workspace/m3/runs/m3_v2/{f}', '/workspace/m3/resultado')
shutil.copy('/workspace/m3/comparacion.txt', '/workspace/m3/resultado')
print(shutil.make_archive('/workspace/m3_resultado', 'zip', '/workspace/m3/resultado'))